# 📚 Полная теория: Регрессия на примере House Prices

## Уровень детализации: 100/10

---

# Часть 1. Введение в регрессионный анализ

## 1.1. Что такое регрессия?

**Регрессия** — это класс задач машинного обучения с учителем, в которых целевая переменная является **непрерывной величиной**. Модель учится предсказывать число, а не метку класса.

**Ключевое отличие от классификации:**

| Аспект | Регрессия | Классификация |
|--------|-----------|---------------|
| Целевая переменная | Непрерывная (цена, температура, рост) | Дискретная (класс, метка) |
| Пример | SalePrice = $250,000 | Survived = 0 или 1 |
| Метрики | RMSE, MAE, R² | Accuracy, Precision, Recall, F1 |
| Выход модели | Число из диапазона | Вероятность класса |

**Математическая постановка:**

Дано:
- Обучающая выборка: $D = \{(x_1, y_1), (x_2, y_2), ..., (x_n, y_n)\}$
- $x_i \in \mathbb{R}^p$ — вектор признаков (p признаков)
- $y_i \in \mathbb{R}$ — целевая переменная

Найти:
- Функцию $f: \mathbb{R}^p \to \mathbb{R}$, которая минимизирует ошибку предсказания

---

## 1.2. Датасет House Prices — Ames Housing Dataset

**Источник:** Kaggle Competition «House Prices - Advanced Regression Techniques»

**Историческая справка:** Датасет создан **Дином Де Коком (Dean De Cock)** в 2011 году как современная замена устаревшему Boston Housing Dataset. Содержит данные о жилых домах в **Эймсе, штат Айова, США**, проданных в период с 2006 по 2010 год.

**Структура:**

| Параметр | Значение |
|----------|----------|
| Всего наблюдений | 2919 (1460 train + 1459 test) |
| Всего признаков | 79 explanatory variables + Id + SalePrice |
| Числовые признаки | 36 (int64, float64) |
| Категориальные признаки | 43 (object) |
| Целевая переменная | `SalePrice` — цена продажи в долларах |

**Ключевые признаки:**

| Признак | Тип | Описание | Влияние на цену |
|---------|-----|----------|-----------------|
| `OverallQual` | int (1-10) | Общее качество материалов и отделки | ⭐⭐⭐⭐⭐ (r ≈ 0.79) |
| `GrLivArea` | float | Жилая площадь над землёй (кв. футы) | ⭐⭐⭐⭐⭐ (r ≈ 0.71) |
| `GarageCars` | int (0-4) | Вместимость гаража (машин) | ⭐⭐⭐⭐ (r ≈ 0.64) |
| `GarageArea` | float | Площадь гаража (кв. футы) | ⭐⭐⭐⭐ (r ≈ 0.62) |
| `TotalBsmtSF` | float | Общая площадь подвала | ⭐⭐⭐⭐ (r ≈ 0.61) |
| `1stFlrSF` | float | Площадь первого этажа | ⭐⭐⭐⭐ (r ≈ 0.61) |
| `YearBuilt` | int | Год постройки | ⭐⭐⭐ (r ≈ 0.52) |
| `FullBath` | int | Полные ванные комнаты | ⭐⭐⭐ (r ≈ 0.56) |
| `TotRmsAbvGrd` | int | Всего комнат над землёй | ⭐⭐⭐ (r ≈ 0.53) |
| `Fireplaces` | int | Количество каминов | ⭐⭐⭐ (r ≈ 0.47) |
| `Neighborhood` | категория (25) | Район в Эймсе | ⭐⭐⭐⭐ (сильное влияние) |
| `ExterQual` | порядковый | Качество внешней отделки | ⭐⭐⭐⭐ |

**Особенности датасета, делающие его идеальным для обучения:**

1. **Реалистичные пропуски** — не случайные, а семантически осмысленные
2. **Скошенное распределение целевой** — требует логарифмирования
3. **Множество категориальных признаков** — 43 из 81 столбца
4. **Мультиколлинеарность** — GarageCars ↔ GarageArea (r = 0.88)
5. **Выбросы** — элитные дома и дома с аномально большой площадью
6. **Риск переобучения** — 79 признаков при 1460 наблюдениях

---

# Часть 2. Метод наименьших квадратов (OLS)

## 2.1. Линейная регрессия

**Модель линейной регрессии** предполагает, что целевая переменная линейно зависит от признаков:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + ... + w_p x_p$$

где:
- $w_0$ — смещение (bias, intercept)
- $w_1, ..., w_p$ — веса (коэффициенты)
- $\hat{y}$ — предсказание модели

**В матричной форме:**

$$\hat{\mathbf{y}} = \mathbf{X}\mathbf{w}$$

где $\mathbf{X} \in \mathbb{R}^{n \times (p+1)}$ — матрица признаков (с добавленным столбцом единиц для bias), $\mathbf{w} \in \mathbb{R}^{p+1}$ — вектор весов.

## 2.2. Метод наименьших квадратов (Ordinary Least Squares, OLS)

**Идея:** найти такие веса $\mathbf{w}$, при которых сумма квадратов отклонений предсказаний от истинных значений минимальна.

**Функция потерь (MSE — Mean Squared Error):**

$$L(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{1}{n} \sum_{i=1}^{n} \left(y_i - \sum_{j=0}^{p} w_j x_{ij}\right)^2$$

**В матричной форме:**

$$L(\mathbf{w}) = \frac{1}{n} \|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2 = \frac{1}{n} (\mathbf{y} - \mathbf{X}\mathbf{w})^T(\mathbf{y} - \mathbf{X}\mathbf{w})$$

**Аналитическое решение (нормальное уравнение):**

Чтобы найти минимум, берём производную по $\mathbf{w}$ и приравниваем к нулю:

$$\frac{\partial L}{\partial \mathbf{w}} = -\frac{2}{n} \mathbf{X}^T(\mathbf{y} - \mathbf{X}\mathbf{w}) = 0$$

$$\mathbf{X}^T\mathbf{X}\mathbf{w} = \mathbf{X}^T\mathbf{y}$$

$$\hat{\mathbf{w}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

**Свойства OLS-оценки:**

| Свойство | Описание |
|----------|----------|
| **Несмещённость** | $E[\hat{\mathbf{w}}] = \mathbf{w}_{true}$ при выполнении предпосылок |
| **Эффективность** | Наименьшая дисперсия среди всех несмещённых линейных оценок (теорема Гаусса-Маркова) |
| **Состоятельность** | $\hat{\mathbf{w}} \to \mathbf{w}_{true}$ при $n \to \infty$ |

**Предпосылки OLS (теорема Гаусса-Маркова):**

1. **Линейность:** $y_i = \mathbf{w}^T\mathbf{x}_i + \varepsilon_i$
2. **Случайная выборка:** $(x_i, y_i)$ независимы
3. **Нулевое условное среднее:** $E[\varepsilon_i | \mathbf{x}_i] = 0$
4. **Гомоскедастичность:** $Var(\varepsilon_i | \mathbf{x}_i) = \sigma^2$ (постоянная дисперсия)
5. **Отсутствие мультиколлинеарности:** $\mathbf{X}^T\mathbf{X}$ обратима

**Проблемы OLS на датасете House Prices:**

| Проблема | Причина | Последствие |
|----------|---------|-------------|
| Переобучение | 79 признаков при 1460 наблюдениях | Плохое обобщение на test |
| Мультиколлинеарность | GarageCars ↔ GarageArea (0.88) | Нестабильные веса |
| Выбросы | Элитные дома, аномальная площадь | Сильное искажение |
| Скошенность | SalePrice skew = 1.88 | Нарушение нормальности остатков |

**Именно эти проблемы решают методы регуляризации и предобработки.**

---

# Часть 3. Регуляризация: Ridge, Lasso, ElasticNet

## 3.1. Зачем нужна регуляризация?

При большом числе признаков OLS-решение переобучается: модель запоминает шум в обучающих данных, а на новых данных показывает плохой результат.

**Идея регуляризации:** добавить штраф за большие веса к функции потерь. Это заставляет модель искать **простое** решение.

## 3.2. Ridge-регрессия (L2-регуляризация)

**Функция потерь:**

$$L_{Ridge}(\mathbf{w}) = \underbrace{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}_{\text{MSE}} + \underbrace{\alpha \sum_{j=1}^{p} w_j^2}_{L_2\text{-штраф}}$$

где $\alpha > 0$ — гиперпараметр,控制 силу регуляризации.

**Аналитическое решение:**

$$\hat{\mathbf{w}}_{Ridge} = (\mathbf{X}^T\mathbf{X} + \alpha \mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$

Добавление $\alpha \mathbf{I}$ гарантирует обратимость матрицы даже при мультиколлинеарности.

**Свойства Ridge:**

- Веса **уменьшаются**, но **не обнуляются**
- Хорошо работает при мультиколлинеарности
- Не выполняет отбор признаков
- Эквивалентна MAP-оценке с гауссовским априорным распределением весов: $w_j \sim \mathcal{N}(0, \frac{1}{2\alpha})$

**Влияние $\alpha$:**

| $\alpha$ | Эффект |
|----------|--------|
| $\alpha \to 0$ | OLS (переобучение) |
| $\alpha$ оптимальное | Баланс bias-variance |
| $\alpha \to \infty$ | Все веса $\to 0$ (недообучение) |

## 3.3. Lasso-регрессия (L1-регуляризация)

**Функция потерь:**

$$L_{Lasso}(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |w_j|$$

**Ключевое отличие от Ridge:** L1-штраф заставляет некоторые веса **точно обнуляться**. Это происходит потому, что контур L1-штрафа имеет «углы» на осях, и минимум часто попадает именно в угол.

**Свойства Lasso:**

- Выполняет **автоматический отбор признаков** (feature selection)
- Разреженное решение (sparse solution)
- Устойчив к выбросам в признаках
- Эквивалентна MAP-оценке с лапласовским априорным распределением: $w_j \sim \text{Laplace}(0, 1/\alpha)$

**Геометрическая интерпретация:**

```
Ridge: минимизация MSE при условии ||w||₂² ≤ t   (круг)
Lasso: минимизация MSE при условии ||w||₁ ≤ t    (ромб)
```

Из-за формы ромба оптимум часто на вершине → некоторые $w_j = 0$.

## 3.4. ElasticNet — компромисс

**Функция потерь:**

$$L_{ElasticNet}(\mathbf{w}) = \text{MSE} + \alpha \left( \rho \sum_{j=1}^{p} |w_j| + \frac{1-\rho}{2} \sum_{j=1}^{p} w_j^2 \right)$$

где:
- $\alpha$ — общая сила регуляризации
- $\rho$ — баланс между L1 и L2 (0 ≤ ρ ≤ 1)

**Свойства:**

- Комбинирует отбор признаков (L1) и устойчивость (L2)
- Работает лучше Lasso при сильно коррелированных признаках
- Требует подбора двух гиперпараметров

## 3.5. Сравнение методов

| Метод | Штраф | Отбор признаков | Устойчивость к мультиколлинеарности | Гиперпараметры |
|-------|-------|-----------------|-------------------------------------|----------------|
| OLS | — | ❌ | ❌ | — |
| Ridge | L2 | ❌ | ✅ | α |
| Lasso | L1 | ✅ | ⚠️ (нестабилен) | α |
| ElasticNet | L1 + L2 | ✅ | ✅ | α, ρ |

---

# Часть 4. Предобработка данных для регрессии

## 4.1. Пропуски в данных

**Типы пропусков (по механизму возникновения):**

| Тип | Аббревиатура | Описание | Пример в House Prices |
|-----|--------------|----------|----------------------|
| Полностью случайные | MCAR | Пропуск не зависит ни от чего | Редкий случай |
| Случайные | MAR | Пропуск зависит от других признаков | LotFrontage зависит от Neighborhood |
| Неслучайные | MNAR | Пропуск зависит от самого значения | PoolQC = NaN, потому что нет бассейна |

**Стратегии обработки:**

### 4.1.1. Смысловые пропуски → "None" или 0

В House Prices пропуски в PoolQC, GarageType, BsmtQual **не являются потерей данных** — они означают «отсутствие объекта».

```python
# PoolQC = NaN → нет бассейна → "None"
train_df['PoolQC'] = train_df['PoolQC'].fillna('None')
# GarageArea = NaN → нет гаража → 0
train_df['GarageArea'] = train_df['GarageArea'].fillna(0)
```

### 4.1.2. Случайные пропуски → медиана по группам

LotFrontage — протяжённость участка вдоль улицы. Зависит от района:

```python
train_df['LotFrontage'] = train_df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
```

### 4.1.3. KNN-импутация

Заполнение на основе K ближайших соседей в пространстве признаков:

```python
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(X)
```

**Почему KNN может быть лучше:** учитывает нелинейные связи между признаками.

## 4.2. Обнаружение и обработка выбросов

### 4.2.1. Метод IQR (Interquartile Range)

**Формула:**

$$IQR = Q_3 - Q_1$$

$$\text{Нижняя граница} = Q_1 - 1.5 \cdot IQR$$

$$\text{Верхняя граница} = Q_3 + 1.5 \cdot IQR$$

**Интерпретация:** объекты за границами считаются выбросами. Для нормального распределения за границами лежит ~0.7% данных.

**Преимущества:** не требует предположения о нормальности, устойчив к выбросам.

### 4.2.2. Метод Z-score

**Формула:**

$$z_i = \frac{x_i - \mu}{\sigma}$$

**Правило:** $|z_i| > 3$ → выброс.

**Ограничение:** предполагает нормальное распределение. Для скошенных данных (как SalePrice) работает плохо.

### 4.2.3. Модифицированный Z-score

Для устойчивости к выбросам используют медиану и MAD:

$$M_i = \frac{0.6745 \cdot (x_i - \text{median})}{\text{MAD}}$$

где MAD — median absolute deviation.

**Правило:** $|M_i| > 3.5$ → выброс.

### 4.2.4. Обработка выбросов в House Prices

В соревновании Kaggle известны конкретные выбросы: дома с GrLivArea > 4000 кв. футов и SalePrice < $300,000. Это аномалии — большой дом по низкой цене.

```python
train_df = train_df[
    (train_df['GrLivArea'] < 4000) & 
    (train_df['SalePrice'] < 500000)
]
```

**Важно:** удалять выбросы нужно **только из train**, не из test! Иначе утечка данных.

## 4.3. Логарифмирование скошенной целевой переменной

### 4.3.1. Проблема скошенности

SalePrice имеет **правую асимметрию** (skew ≈ 1.88):

- Большинство домов дешёвые
- Несколько домов очень дорогие
- Распределение не нормальное

**Последствия для регрессии:**

1. Нарушение предпосылки нормальности остатков
2. Гетероскедастичность (дисперсия ошибок растёт с ростом y)
3. Большое влияние выбросов
4. Смещённые предсказания для дорогих домов

### 4.3.2. Логарифмическое преобразование

**Формула:**

$$y' = \log(1 + y) \quad \text{(log1p)}$$

**Обратное преобразование:**

$$y = \exp(y') - 1 \quad \text{(expm1)}$$

**Почему log1p, а не log?** log1p определён при y = 0, log(0) = -∞.

**Эффект:**

| До log1p | После log1p |
|----------|-------------|
| skew = 1.88 | skew ≈ 0.12 |
| Правая асимметрия | Почти нормальное |
| Гетероскедастичность | Гомоскедастичность |

### 4.3.3. Интерпретация коэффициентов

После логарифмирования целевой переменной коэффициенты интерпретируются как **относительные изменения**:

$$\frac{\Delta y}{y} \approx \Delta \log(y) = w_j \cdot \Delta x_j$$

Например, если $w_j = 0.05$ для OverallQual, то увеличение OverallQual на 1 балл связано с ростом цены на ~5%.

### 4.3.4. Важное предостережение

Метрики, посчитанные в log-пространстве (RMSE log), **не равны** метрикам в долларах. Для интерпретации нужно возвращать предсказания в исходное пространство:

```python
y_pred_log = model.predict(X_test)
y_pred_real = np.expm1(y_pred_log)
```

## 4.4. Кодирование категориальных признаков

### 4.4.1. Label Encoding (порядковое)

Для **порядковых** признаков, где важен порядок:

```python
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
train_df['ExterQual'] = train_df['ExterQual'].map(qual_map)
```

### 4.4.2. One-Hot Encoding (номинальное)

Для **номинальных** признаков без порядка:

```python
train_df = pd.get_dummies(train_df, columns=['MSZoning'], drop_first=True)
```

**drop_first=True** — удаляем первую категорию, чтобы избежать мультиколлинеарности (dummy variable trap).

### 4.4.3. Target Encoding

Замена категории на среднее значение целевой переменной:

$$\text{Neighborhood}_k \to \frac{1}{n_k} \sum_{i \in \text{Neighborhood}_k} y_i$$

**Риск утечки данных:** нужно вычислять внутри кросс-валидации.

### 4.4.4. Сравнение методов для House Prices

| Признак | Тип | Метод | Обоснование |
|---------|-----|-------|-------------|
| ExterQual | Порядковый | Label | Порядок важен |
| Neighborhood | Номинальный (25) | Target Encoding | Много категорий |
| MSZoning | Номинальный (5) | One-Hot | Мало категорий |
| GarageType | Номинальный (6) | One-Hot | Умеренно |

## 4.5. Масштабирование признаков

### 4.5.1. StandardScaler

$$z = \frac{x - \mu}{\sigma}$$

Результат: mean = 0, std = 1.

**Когда:** нормальное распределение, LR, Ridge, Lasso, SVM.

### 4.5.2. MinMaxScaler

$$z = \frac{x - \min}{\max - \min}$$

Результат: диапазон [0, 1].

**Когда:** нейросети, CNN. Чувствителен к выбросам.

### 4.5.3. RobustScaler

$$z = \frac{x - \text{median}}{IQR}$$

**Когда:** много выбросов.

### 4.5.4. Что масштабировать?

| Модель | Нужно масштабирование? |
|--------|------------------------|
| Linear Regression | ✅ Да |
| Ridge, Lasso, ElasticNet | ✅ Да (для корректной регуляризации) |
| Decision Tree | ❌ Нет |
| Random Forest | ❌ Нет |
| XGBoost | ❌ Нет |
| SVM | ✅ Обязательно |
| KNN | ✅ Обязательно |

---

# Часть 5. Feature Engineering

## 5.1. Создание новых признаков

Feature Engineering — искусство создания признаков, которые лучше описывают предметную область.

### 5.1.1. Суммирование и агрегация

```python
# Общая площадь дома
train_df['TotalSF'] = (
    train_df['TotalBsmtSF'] +    # подвал
    train_df['1stFlrSF'] +        # первый этаж
    train_df['2ndFlrSF']           # второй этаж
)

# Общее число ванных
train_df['TotalBath'] = (
    train_df['FullBath'] + 
    0.5 * train_df['HalfBath'] +   # половинные = 0.5
    train_df['BsmtFullBath'] + 
    0.5 * train_df['BsmtHalfBath']
)

# Общее крыльцо
train_df['TotalPorch'] = (
    train_df['OpenPorchSF'] + 
    train_df['EnclosedPorch'] + 
    train_df['3SsnPorch'] + 
    train_df['ScreenPorch'] + 
    train_df['WoodDeckSF']
)
```

### 5.1.2. Возрастные признаки

```python
# Возраст дома на момент продажи
train_df['HouseAge'] = train_df['YrSold'] - train_df['YearBuilt']

# Возраст последней реконструкции
train_df['RemodAge'] = train_df['YrSold'] - train_df['YearRemodAdd']

# Был ли дом реконструирован?
train_df['IsRemodeled'] = (train_df['YearRemodAdd'] != train_df['YearBuilt']).astype(int)
```

### 5.1.3. Бинарные признаки

```python
train_df['HasPool'] = (train_df['PoolArea'] > 0).astype(int)
train_df['Has2ndFloor'] = (train_df['2ndFlrSF'] > 0).astype(int)
train_df['HasGarage'] = (train_df['GarageArea'] > 0).astype(int)
train_df['HasBsmt'] = (train_df['TotalBsmtSF'] > 0).astype(int)
train_df['HasFireplace'] = (train_df['Fireplaces'] > 0).astype(int)
```

### 5.1.4. Взаимодействия

```python
# Качество × площадь — комбинированный эффект
train_df['OverallQual_TotalSF'] = train_df['OverallQual'] * train_df['TotalSF']

# Качество × возраст
train_df['Qual_Age'] = train_df['OverallQual'] / (train_df['HouseAge'] + 1)
```

### 5.1.5. Полиномиальные признаки

```python
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
```

**Осторожно:** полиномиальные признаки быстро увеличивают размерность и риск переобучения.

## 5.2. Отбор признаков

### 5.2.1. Методы фильтрации

- **Корреляция Пирсона** — линейная связь
- **Mutual Information** — любая зависимость
- **Variance Threshold** — удаление почти константных признаков

### 5.2.2. Методы обёртки

- **RFE (Recursive Feature Elimination)** — рекурсивное удаление наименее важных
- **Forward Selection** — добавление по одному
- **Backward Elimination** — удаление по одному

### 5.2.3. Встроенные методы

- **Lasso** — обнуляет неинформативные признаки
- **Tree-based importance** — важность из Random Forest / XGBoost

---

# Часть 6. Оценка качества регрессионных моделей

## 6.1. Метрики

### 6.1.1. MSE (Mean Squared Error)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Свойства:**
- Штрафует большие ошибки сильнее (квадрат)
- Чувствительна к выбросам
- Единицы: квадрат целевой переменной

### 6.1.2. RMSE (Root Mean Squared Error)

$$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

**Свойства:**
- В тех же единицах, что и целевая
- Интерпретируема: «средняя ошибка в долларах»
- Чувствительна к выбросам

### 6.1.3. MAE (Mean Absolute Error)

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Свойства:**
- Устойчива к выбросам
- В тех же единицах, что и целевая
- Соответствует медианной ошибке

### 6.1.4. R² (Coefficient of Determination)

$$R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2} = 1 - \frac{\text{SS}_{res}}{\text{SS}_{tot}}$$

**Интерпретация:**
- $R^2 = 1$ — идеальное предсказание
- $R^2 = 0$ — модель не лучше среднего
- $R^2 < 0$ — модель хуже среднего

**Свойства:**
- Безразмерная (сравнение между датасетами)
- Не зависит от масштаба целевой
- Может быть обманчива при переобучении

### 6.1.5. Adjusted R²

$$R^2_{adj} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

где n — число наблюдений, p — число признаков.

**Зачем:** штрафует за добавление бесполезных признаков. $R^2$ всегда растёт при добавлении признаков, Adjusted R² — нет.

## 6.2. Сравнение метрик для House Prices

| Метрика | Значение | Интерпретация |
|---------|----------|---------------|
| RMSE (log) | 0.12 | Средняя ошибка в log-пространстве |
| RMSE ($) | ~$25,000 | Средняя ошибка в долларах |
| MAE ($) | ~$17,000 | Медианная ошибка |
| R² | 0.92 | Модель объясняет 92% дисперсии |

**Что важнее для бизнеса?** Для продажи недвижимости часто важнее MAE — типичная ошибка, а не выбросы.

---

# Часть 7. Кросс-валидация и подбор гиперпараметров

## 7.1. Проблема одного train_test_split

При единственном разделении:
- Оценка зависит от случайности
- Маленький датасет → нестабильные метрики
- Нет информации о разбросе качества

## 7.2. K-Fold кросс-валидация

**Алгоритм:**
1. Разбить данные на K фолдов
2. Для i = 1..K:
   - Обучить на K-1 фолдах
   - Оценить на i-м фолде
3. Усреднить метрики

**Формула средней оценки:**

$$\text{CV}_K = \frac{1}{K} \sum_{i=1}^{K} \text{Metric}_i$$

**Стандартное отклонение:**

$$\sigma = \sqrt{\frac{1}{K-1} \sum_{i=1}^{K} (\text{Metric}_i - \text{CV}_K)^2}$$

**Преимущества:**
- Каждый объект побывает в test ровно один раз
- Устойчивая оценка
- Можно оценить дисперсию

## 7.3. Стратифицированная кросс-валидация

Для регрессии стратификация обычно не применяется, но для классификации — обязательна.

Для регрессии используют **KBinsDiscretizer** для создания «страт» по целевой переменной.

## 7.4. GridSearchCV

**Алгоритм:**
1. Задать сетку параметров
2. Для каждой комбинации:
   - Провести K-Fold CV
   - Записать среднюю метрику
3. Выбрать комбинацию с лучшей метрикой

**Пример для Ridge:**

```python
param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring='neg_rmse')
grid.fit(X_train, y_train)
print(grid.best_params_)  # {'alpha': 1.0}
```

**Проблема:** экспоненциальный рост числа комбинаций.

## 7.5. RandomizedSearchCV

Случайный выбор N комбинаций из распределений:

```python
from scipy.stats import loguniform
param_dist = {'alpha': loguniform(1e-4, 1e2)}
random_search = RandomizedSearchCV(Ridge(), param_dist, n_iter=50, cv=5)
```

**Преимущества:** быстрее GridSearch, часто находит оптимум.

---

# Часть 8. Диагностика регрессионных моделей

## 8.1. Анализ остатков

**Остатки:** $e_i = y_i - \hat{y}_i$

**Что проверять:**

1. **Среднее остатков ≈ 0** — нет систематической ошибки
2. **Нормальность** — QQ-plot
3. **Гомоскедастичность** — разброс не меняется
4. **Отсутствие автокорреляции** — Durbin-Watson

**График остатков:**

```python
plt.scatter(y_pred, residuals)
plt.axhline(y=0, color='r', linestyle='--')
```

**Паттерны:**

| Паттерн | Проблема |
|---------|----------|
| Случайный разброс | ✅ Всё хорошо |
| Веер | Гетероскедастичность |
| U-образный | Нелинейность |
| Тренд | Смещение |

## 8.2. Кривые обучения

**График:** train error и validation error vs размер обучающей выборки.

```python
from sklearn.model_selection import learning_curve
train_sizes, train_scores, val_scores = learning_curve(
    estimator, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10)
)
```

**Интерпретация:**

| Ситуация | Диагноз | Решение |
|----------|---------|---------|
| Train низкий, Val высокий | Переобучение | Регуляризация, больше данных |
| Train высокий, Val высокий | Недообучение | Сложнее модель, больше признаков |
| Обе низкие, сходятся | Хорошо | Оптимум |

## 8.3. Кривые валидации

**График:** метрика vs гиперпараметр (например, α для Ridge).

```python
from sklearn.model_selection import validation_curve
train_scores, val_scores = validation_curve(
    Ridge(), X, y, param_name='alpha', param_range=[0.001, 0.01, 0.1, 1, 10]
)
```

**Помогает найти оптимум α** и увидеть, где начинается переобучение.

---

# Часть 9. Практические аспекты для House Prices

## 9.1. Полный пайплайн предобработки

```
1. Загрузка данных
2. EDA: распределения, корреляции, пропуски
3. Обработка пропусков:
   - Смысловые → "None" или 0
   - Случайные → медиана по группам
   - Оставшиеся → мода
4. Обработка выбросов: IQR или доменные знания
5. Логарифмирование SalePrice
6. Feature Engineering:
   - TotalSF, TotalBath, TotalPorch
   - HouseAge, RemodAge
   - HasPool, HasGarage
   - Взаимодействия
7. Кодирование категорий:
   - Порядковые → Label
   - Номинальные → One-Hot или Target
8. Разделение train/test
9. Масштабирование (fit на train)
10. Обучение моделей: LR, Ridge, Lasso, ElasticNet
11. Кросс-валидация и GridSearchCV
12. Оценка на test
13. Интерпретация коэффициентов
14. Сохранение модели
```

## 9.2. Что даёт каждый шаг предобработки

| Шаг | Влияние на RMSE (log) | Обоснование |
|-----|----------------------|-------------|
| Baseline (без обработки) | ~0.20 | Пропуски, выбросы |
| + Обработка пропусков | ~0.16 | Убрали NaN |
| + Логарифмирование целевой | ~0.14 | Нормальность |
| + Feature Engineering | ~0.13 | Больше информации |
| + Регуляризация (Lasso) | ~0.12 | Борьба с переобучением |
| + Кросс-валидация + GridSearch | ~0.115 | Оптимальные гиперпараметры |

## 9.3. Типичные ошибки

| Ошибка | Последствие | Правильно |
|--------|-------------|-----------|
| `scaler.fit_transform(X_all)` до split | Утечка данных | Сначала split |
| Удаление выбросов из test | Утечка | Только из train |
| Target Encoding до split | Утечка | Внутри CV |
| Подача категорий строками в модель | Ошибка | Кодировать |
| Игнор мультиколлинеарности | Нестабильные веса | Ridge/Lasso |
| Обучение на скошенной целевой | Плохой R² | log1p |

---

# Часть 10. Итоговые выводы

## 10.1. Что важно запомнить

1. **Регрессия** — предсказание непрерывной величины. **Классификация** — дискретной.

2. **OLS** минимизирует MSE. Аналитическое решение: $\hat{\mathbf{w}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$.

3. **Ridge** (L2) уменьшает веса, **Lasso** (L1) обнуляет их — выполняет отбор признаков.

4. **ElasticNet** — компромисс между Ridge и Lasso.

5. **Логарифмирование** скошенной целевой переменной улучшает нормальность и гомоскедастичность.

6. **Пропуски** бывают смысловыми (→ "None") и случайными (→ медиана).

7. **Выбросы** удаляются методом IQR или Z-score. Только из train!

8. **Кросс-валидация** даёт устойчивую оценку. **GridSearchCV** — оптимальные гиперпараметры.

9. **fit_transform** только на train. **transform** на test.

10. **R²** — доля объяснённой дисперсии. **RMSE** — средняя ошибка в единицах целевой.

## 10.2. Чек-лист для любого регрессионного проекта

```
□ Проверил распределение целевой (skew?)
□ Обработал пропуски (семантика + статистика)
□ Удалил выбросы (IQR/Z-score, только train)
□ Логарифмировал скошенную целевую (если нужно)
□ Создал новые признаки (агрегации, взаимодействия)
□ Закодировал категориальные (Label/One-Hot/Target)
□ Разделил train/test ДО масштабирования
□ Масштабировал признаки (fit на train)
□ Обучил baseline модель
□ Сравнил Ridge/Lasso/ElasticNet
□ Провёл кросс-валидацию
□ Подобрал гиперпараметры (GridSearchCV)
□ Оценил на test (RMSE, MAE, R²)
□ Проанализировал остатки
□ Интерпретировал коэффициенты
□ Сохранил модель и метаданные
```

---

**Ключевой вывод:** House Prices — идеальный датасет для изучения регрессии, потому что он содержит все классические проблемы: пропуски, выбросы, скошенность, мультиколлинеарность, категориальные признаки. Решение этих проблем через предобработку и регуляризацию — фундаментальный навык ML-инженера.